# Exposure Metrics

This notebook creates the Exposure section CSV for the national tool. The seven selectable cards are **Population**, **Capital Stock**, **Roads**, **Rail**, **Power**, **Hospitals**, and **Schools**.

## 0. Setup

Country settings, source paths, and output conventions come from `config/countries/KEN.toml`. Shared calculations live in `src/national_tool_metrics/` so they can be reused by other section notebooks.

In [ ]:
from pathlib import Path
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import (
    build_identifier_frame,
    merge_metric_tables,
    validate_section_output,
    write_section_output,
)
from national_tool_metrics.sections.exposure import (
    build_capital_stock_metrics,
    build_facility_metrics,
    build_network_metrics,
    build_population_metrics,
)

In [ ]:
config = load_country_config("KEN", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
admin_regions[["adm_id", "adm_name"]].head()

## 1. Population

Summarize total population and the agreed sex and age breakdowns from WorldPop rasters. Youth aged 15–24 is not included.

In [ ]:
population_metrics = build_population_metrics(config, admin_regions)
population_metrics.head()

## 2. Capital Stock

Summarize residential, non-residential, and infrastructure capital stock within each administrative region.

In [ ]:
capital_stock_metrics = build_capital_stock_metrics(config, admin_regions)
capital_stock_metrics.head()

## 3. Roads, Rail, and Power

Clip network lines to administrative boundaries and summarize road length by type, total rail length, and total power-transmission length.

In [ ]:
network_metrics = build_network_metrics(config, admin_regions)
network_metrics.head()

## 4. Hospitals and Schools

Load the existing administrative summaries of government hospital and school buildings.

In [ ]:
facility_metrics = build_facility_metrics(config, admin_regions)
facility_metrics.head()

## 5. Combine and Validate

Combine the seven card groups into one row per administrative region using the shared section-output contract.

In [ ]:
identifiers = build_identifier_frame(
    admin_regions,
    config,
    section="exposure",
    hazard="none",
    scenario="baseline",
    model_run="baseline_inputs",
)

exposure_metrics = merge_metric_tables(
    identifiers,
    [
        population_metrics,
        capital_stock_metrics,
        network_metrics,
        facility_metrics,
    ],
)
validate_section_output(exposure_metrics, "exposure")

print(f"Rows: {len(exposure_metrics):,}")
print(f"Metric columns: {len(exposure_metrics.columns) - len(identifiers.columns):,}")
exposure_metrics.head()

## 6. Export

Write the canonical Exposure CSV only after reviewing the combined table above.

In [ ]:
output_path = write_section_output(exposure_metrics, config, "exposure")
print(f"Exported Exposure metrics to: {output_path}")